<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
<br>汉化的库: <a href="https://github.com/GoatCsu/CN-LLMs-from-scratch.git">https://github.com/GoatCsu/CN-LLMs-from-scratch.git</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 第二章:处理文本数据 

本章节需要安装的包

pip3 install importlib.metadata

In [213]:
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))  # tiktoken: GPT 专用分词器
# 确认库已安装并显示当前安装的版本

# BPE 是目前绝大多数主流大模型（GPT、LLaMA、Claude 等）的核心分词算法，tiktoken 就是 OpenAI 基于 BPE 实现的工具

torch version: 2.11.0+cu128
tiktoken version: 0.12.0


- 本章节已经为LLM的实现构建了数据库

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/01.webp?timestamp=1" width="500px">

## 2.1 理解文字embedding

- 无代码

- 在众多形式的embedding中,我们只讨论text embedding(文本样本)
- embedding含义丰富,而且是常用词汇,所以以下皆不做翻译,多加体会!

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/02.webp" width="500px">

- LLM从高纬空间视角理解文字(i.e., 上千个dimension)
- 虽然人类只能想象低维的视角,我们无法描绘计算机所理解的embedding
- 但是下图我们粗浅的从二维上模拟计算机的视角

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/03.webp" width="300px">

## 2.2 文本标签化(tokenize)

- 关于embedding token,实在是不好翻译,于是有时候我会选择不去翻译这两个专有名词
- 本节中,我们将tokenize文字信息. 这会把文字拆解为更多小的理解单元 例如单个词或者字节

(这也有点抽象，事实上可以粗略理解为将单词拆分为词根、词源和词缀)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/04.webp" width="300px">

- 载入源文件
- [The Verdict by Edith Wharton](https://en.wikisource.org/wiki/The_Verdict) 一本无版权的短篇小说

In [214]:
import os ##导入os库
import urllib.request ##导入request库: Python 内置的 HTTP 请求库，用来从网络上下载文件
# 如果文件不存在则创建，避免重复下载
if not os.path.exists("the-verdict.txt"): # 检查当前工作目录下，是否已经存在名为 the-verdict.txt 的文件，如果文件不存在，才执行下面的下载逻辑
    #url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"
    url = ("https://raw.githubusercontent.com/rasbt/" # 拼接文件下载地址
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
           "the-verdict.txt")     # 这个地址是作者 Sebastian Raschka 在 GitHub 上公开的示例文本文件，内容是短篇小说《The Verdict》，作为后续分词、训练的示例语料
    file_path = "the-verdict.txt" # 把下载后的文件，保存在当前工作目录下，文件名也叫 the-verdict.txt
    urllib.request.urlretrieve(url, file_path) # 从指定的地点读取文件

- 如果在执行前面的代码单元时遇到 `ssl.SSLCertVerificationError` 错误可能是由于使用了过时的 Python 版本；
- 你可以在 [GitHub 上查阅更多信息](https://github.com/rasbt/LLMs-from-scratch/pull/403)。

In [215]:
# with open(...) as f: Python 的「上下文管理器」写法，会自动帮你关闭文件，不用手动写 f.close()，更安全，不会出现文件句柄泄漏
with open("the-verdict.txt", "r", encoding="utf-8") as f: # 要打开的文件名 / "r"：打开模式「只读」 / 指定用 UTF-8 编码读取文件，避免中文和特殊字符乱码
    raw_text = f.read() # 一次性读取文件的全部内容，存到 raw_text 变量里，变成一个超长字符串

print("文本总字符数:", len(raw_text))##先输出总长度
print(raw_text[:99])##输出前一百个内容

文本总字符数: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


- 目标是对这段文本进行分词和嵌入处理，以便用于大语言模型（LLM）。
- 我们将基于一些简单的示例文本开发一个简单的分词器，之后可以将其应用于上述文本。
- 以下正则表达式将基于空格进行分割。

In [216]:
import re  # re 是 Python 内置的正则表达式库，用来处理字符串的匹配、分割、替换等操作

text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)##正则表达式按照空白字符进行分割

print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


- 优化正则表达,可以分割更多的符号

In [217]:
result = re.split(r'([,.]|\s)', text)##只是按照, .分割
print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


- 移除空格

In [218]:
##把上述结果去掉空格
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


- 我们还需要处理其他标点符号

In [219]:
text = "Hello, world. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', text) ##就是按照常用的符号分割
result = [item.strip() for item in result if item.strip()]##去掉两端的空白字符 也去掉了空字符串与空白字符项
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


- 万事俱备，我们现在来看一下文字处理后的效果

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/05.webp" width="350px">

In [220]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text) ##按照符号继续把原文件给分割了
preprocessed = [item.strip() for item in preprocessed if item.strip()]##去掉两端的空白字符 也是去掉了空字符串和仅包含空白字符的项
# 这一步之后，列表里就只剩下「纯单词」和「纯标点/符号」的 token 了
print(preprocessed[:30])  # 查看前 30 个 token

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


- 查看token的长度

In [221]:
print(len(raw_text))      # 原始文本的总字符数（包括所有字母、空格、标点）
print(len(preprocessed))  # 经过正则分词后，得到的 token 列表的长度
# 这篇 2 万多字符的小说，用这个正则分词器，被切成了 4690 个独立的 token

20479
4690


## 2.3 给token编号

- 通过如下的embedding层,我们可以给token编号

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/06.webp" width="500px">

- 我们要创建一个表格,给所有的token给映射到不同的标号上

In [222]:
all_words = sorted(set(preprocessed)) # 去重排序
vocab_size = len(all_words)           # 词汇表总长度

print(vocab_size)

1130


In [223]:
vocab = {token:integer for integer,token in enumerate(all_words)}##把word进行编号,再按照 单词/标点 分配索引(有HashList那味道了)
print(vocab)  # 查看整个词汇表 (词:索引)

{'!': 0, '"': 1, "'": 2, '(': 3, ')': 4, ',': 5, '--': 6, '.': 7, ':': 8, ';': 9, '?': 10, 'A': 11, 'Ah': 12, 'Among': 13, 'And': 14, 'Are': 15, 'Arrt': 16, 'As': 17, 'At': 18, 'Be': 19, 'Begin': 20, 'Burlington': 21, 'But': 22, 'By': 23, 'Carlo': 24, 'Chicago': 25, 'Claude': 26, 'Come': 27, 'Croft': 28, 'Destroyed': 29, 'Devonshire': 30, 'Don': 31, 'Dubarry': 32, 'Emperors': 33, 'Florence': 34, 'For': 35, 'Gallery': 36, 'Gideon': 37, 'Gisburn': 38, 'Gisburns': 39, 'Grafton': 40, 'Greek': 41, 'Grindle': 42, 'Grindles': 43, 'HAD': 44, 'Had': 45, 'Hang': 46, 'Has': 47, 'He': 48, 'Her': 49, 'Hermia': 50, 'His': 51, 'How': 52, 'I': 53, 'If': 54, 'In': 55, 'It': 56, 'Jack': 57, 'Jove': 58, 'Just': 59, 'Lord': 60, 'Made': 61, 'Miss': 62, 'Money': 63, 'Monte': 64, 'Moon-dancers': 65, 'Mr': 66, 'Mrs': 67, 'My': 68, 'Never': 69, 'No': 70, 'Now': 71, 'Nutley': 72, 'Of': 73, 'Oh': 74, 'On': 75, 'Once': 76, 'Only': 77, 'Or': 78, 'Perhaps': 79, 'Poor': 80, 'Professional': 81, 'Renaissance': 82, 'Ri

- 看一下前50个是怎样的

In [224]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break ##遍历前五十个

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


- 接下来,我们将通过一个短文本来感受下,处理后的效果是怎样的

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/07.webp?123" width="500px">

- 现在将所有内容整合到一个分词器类中。

In [225]:
class SimpleTokenizerV1:  # 一个基于固定词表的极简分词器V1版本,核心功能：文本 ↔ 数字ID 的互相转换
    def __init__(self, vocab):
        self.str_to_int = vocab                          # {词: ID}
        self.int_to_str = {i:s for s,i in vocab.items()} # {ID: 词}
    
    def encode(self, text):  # 词 → ID 的正向映射（编码用）
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)                # 用正则分词：把文本按标点、空白、特殊符号分割，保留标点
        preprocessed = [item.strip() for item in preprocessed if item.strip()]  # 清洗分词结果：去掉每个token两端的空白，同时过滤掉 空字符串/纯空格项
        ids = [self.str_to_int[s] for s in preprocessed]  # 把清洗后的token列表，根据词表转换成ID列表
        return ids                                        # 注意：如果词表里没有这个token，会直接报错（这是V1版本的局限）
        
    def decode(self, ids):  # 反向词表：ID → 词 的反向映射（解码用）
        text = " ".join([self.int_to_str[i] for i in ids]) # 把每个ID通过反向词表映射回对应的词/标点，用空格拼接起来
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)    # 使用正则表达式，修正标点前的多余空格（比如把 "hello , world" 变成 "hello, world"）
        return text

- `encode` 函数 将文本转换为标记 ID。
- `decode` 函数 将标记 ID 转换回文本。

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/08.webp?123" width="500px">

- 我们可以使用分词器将文本编码（即分词）为数字。
- 然后，这些整数可以作为大语言模型（LLM）的输入，进行嵌入。

In [226]:
tokenizer = SimpleTokenizerV1(vocab)  # 用前面的 vocab 创造一个实例
text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)  # 按照这个例子里的encode函数处理text
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


- 把数字重新映射回文字

In [227]:
tokenizer.decode(ids)  # 按照这个例子里的decode函数处理text

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [228]:
tokenizer.decode(tokenizer.encode(text))#按照这个例子里的decode函数处理(#按照这个例子里的encode函数处理text)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

## 2.4 添加特殊标记

- 文本的结尾需要特别的符号来表明

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/09.webp?123" width="500px">

- 一些分词器使用特殊标记来帮助大语言模型（LLM）获取额外的上下文信息。
- 其中一些特殊标记包括：
  - `[BOS]`（序列开始）表示文本的开始。
  - `[EOS]`（序列结束）表示文本的结束（通常用于连接多个不相关的文本，例如两个不同的维基百科文章或两本不同的书籍等）。
  - `[PAD]`（填充）如果我们使用大于1的批次大小训练LLM（我们可能会包含不同长度的多篇文本），使用填充标记将较短的文本填充至最长的长度，以确保所有文本具有相同的长度。
- `[UNK]` 用于表示词汇表中没有的词。

- !请注意! GPT-2不需要上述提到的这些标记，它只使用 `<|endoftext|>` 标记。
- `<|endoftext|>` 类似于上述提到的 `[EOS]` 标记。
- GPT 还使用 `<|endoftext|>` 进行填充（因为我们在批量输入训练时通常使用掩码，所以无论这些填充标记是什么，都不会影响模型的训练，因为填充标记不会被关注）。
- GPT-2 不使用 `<UNK>` 标记来表示词汇表外的词；相反，GPT-2 使用字节对编码（BPE）分词器，将单词分解为子词单元，后续将进一步讨论这一点。
- 我们在两个独立文本之间使用 `<|endoftext|>` 标记：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/10.webp" width="500px">

- 看一下接下来会发生什么

In [229]:
tokenizer = SimpleTokenizerV1(vocab)  ##用vocab创造一个实例
text = "Hello, do you like tea. Is this-- a test?"  # 训练集之外的新样本
tokenizer.encode(text)

KeyError: 'Hello'

- 上述操作会产生一个错误，因为单词“Hello”不在词汇表中。
- 为了处理这种情况，我们可以向词汇表中添加类似 `"<|unk|>"` 的特殊标记，用于表示未知词汇。
- 因为我们已经在扩展词汇表，那么我们可以再添加一个标记 `"<|endoftext|>"`，该标记在GPT-2训练中用于表示文本的结束（它也用于连接的文本之间，例如当我们的训练数据集包含多篇文章、书籍等时）。

In [239]:
print('原始独立token:',len(preprocessed))  # 原始 token (去掉了空格,但还有重复的词)
all_tokens = sorted(list(set(preprocessed)))  # set去重,list重新变为列表,然后排序
print('去重后:',len(all_tokens))
all_tokens.extend(["<|endoftext|>", "<|unk|>"]) # 加上未知的表示
print('加上未知表示:',len(all_tokens))
vocab = {token:integer for integer,token in enumerate(all_tokens)}  # 构建词表：给每个 token 分配一个唯一的 ID {token: id}

原始独立token: 4690
去重后: 1130
加上未知表示: 1132


In [240]:
len(vocab.items())

1132

In [242]:
for i, item in enumerate(list(vocab.items())[-5:]):#输出后五个内容与其ID
    print(item)  # 可以看到最后增加了2个ID

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


- 因此增加 `<unk>`不失为一种好的选择

In [243]:
class SimpleTokenizerV2:##版本2.0,启动! (能够处理未知单词)
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()} # {ID: 词}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)#正则化按照标点分类
        preprocessed = [item.strip() for item in preprocessed if item.strip()]#去掉两头与所有空余句
        preprocessed = [item if item in self.str_to_int else "<|unk|>" for item in preprocessed]
        # ↑ 唯一新增代码: 遍历分词后的每个 token，如果词表里没有这个词，就替换成 <|unk|>，而不是直接报错
        ids = [self.str_to_int[s] for s in preprocessed]#映射为整数列表
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

- 用优化后的分词器对文本进行操作

In [246]:
tokenizer = SimpleTokenizerV2(vocab)
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))#用句子分隔符链接两个句子
print(text) # 看看长啥样

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [247]:
tokenizer.encode(text)  # 不会报错了

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

In [249]:
tokenizer.decode(tokenizer.encode(text))
# 1130: <|endoftext|>
# 1131: <|unk|>

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'

## 2.5 字节对编码

- GPT-2 使用字节对编码（BPE）作为其分词器。
- 这种方式允许模型将不在预定义词汇表中的单词分解为更小的子词单元，甚至是单个字符，从而使其能够处理词汇表外的词汇。
- 例如，如果 GPT-2 的词汇表中没有“unfamiliarword”这个单词，它可能会将其分词为 ["unfam", "iliar", "word"]，或者根据训练好的 BPE 合并规则进行其他子词分解。
- 原始的 BPE 分词器可以在这里找到：[https://github.com/openai/gpt-2/blob/master/src/encoder.py](https://github.com/openai/gpt-2/blob/master/src/encoder.py)
- 在本章中，我们使用了 OpenAI 开源的 [tiktoken](https://github.com/openai/tiktoken) 库中的 BPE 分词器，该库在 Rust 中实现了核心算法，以提高计算性能。
- 在 [./bytepair_encoder](../02_bonus_bytepair-encoder) 中创建了一个笔记本，对比了这两种实现的效果（在样本文本上，tiktoken 的速度大约快了 5 倍）。

In [ ]:
# pip install tiktoken

In [250]:
import importlib
import tiktoken

print("tiktoken 版本:", importlib.metadata.version("tiktoken"))#验证下载并输出版本信息

tiktoken 版本: 0.12.0


In [252]:
tokenizer = tiktoken.get_encoding("gpt2")  # 初始化GPT2!
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})  # 输出分词的id,可以允许<|endoftext|>
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [253]:
strings = tokenizer.decode(integers)  # 按照数字解码回去
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


- BPE tokenizers将未知词汇分解为子词和单个字符。

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/11.webp" width="300px">

## 2.6 利用滑动窗口进行数据采样

- 现在我们训练的大语言模型（LLMs）时是一次生成一个单词，因此希望根据训练数据的要求进行准备，使得序列中的下一个单词作为预测目标。

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/12.webp" width="400px">

In [255]:
# 首先使用 BPE 分词器对短篇小说 the verdict 的全文进行分词
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()  # 读取文本文档 赋值给raw_text

enc_text = tokenizer.encode(raw_text)#读入text并编码到enc_text里
print(len(enc_text))   # token 总数
print(len(set(enc_text)))  # 去重后

5145
1416


- 对于每个文本块，我们需要输入和目标。
- 由于我们希望模型预测下一个单词，因此我们要生成目标是将输入右移一个位置后的单词。

In [256]:
enc_sample = enc_text[50:]  # 第51个token的索引是50, 从第五十一个token开始向后取
context_size = 4  # 滑动窗口大小 (用前4个词，预测下一个词)
x = enc_sample[:context_size]     # 取开头4个token作为输入 (等价于enc_sample[0:4])
y = enc_sample[1:context_size+1]  # 从第2个开始取4个作为目标 (等价于enc_sample[1:5])
print(f"输入 x: {x}")
print(f"预测 y:      {y}")

输入 x: [290, 4920, 2241, 287]
预测 y:      [4920, 2241, 287, 257]


- 就像预言家一个晚上只能预言一个玩家,我们的模型一次也只能预测一个单词

In [257]:
'''
i=1：用 [0] 预测 [1]
i=2：用 [0,1] 预测 [2]
i=3：用 [0,1,2] 预测 [3]
i=4：用 [0,1,2,3] 预测 [4]
'''
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "---->", desired)  # 箭头左侧表示大模型接收的输入,箭头右侧代表要预测的目标

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [258]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))  # ID → 词元

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


- 我们将在后续章节中处理下一个单词预测，届时会介绍注意力机制。
- 目前，我们实现一个简单的数据加载器，它遍历输入数据集并返回右移一个位置后的输入和目标。

- 安装并导入 PyTorch（安装提示请参见附录A）。

In [259]:
import torch
print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


- 用滑动窗口法运行,窗口位置每次加一

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/13.webp?123" width="500px">

- 创建数据集和数据加载器，从输入文本数据集中提取文本块。

In [261]:
from torch.utils.data import Dataset, DataLoader  # 导入PyTorch 数据集基类 和 数据加载器
''' GPTDatasetV1 是一个自回归语言模型专用的数据集类，它会:
1.把整段文本用 tiktoken 分词成 token ID
2.用滑动窗口切分成固定长度的训练序列
3.自动生成「输入序列 x」和「错位的目标序列 y」，直接给后面的模型训练用
'''
class GPTDatasetV1(Dataset): # 继承 Dataset 后：你只需要实现 __len__ 和 __getitem__ 两个方法，就能被 DataLoader 直接调用；
                             # DataLoader 会帮你做批量加载、打乱、多线程处理，不用自己写循环取数据，直接用 DataLoader 喂数据就行，适配后面的模型训练。
    def __init__(self, txt, tokenizer, max_length, stride):  # 原始文本/分词器/上下文窗口大小/滑动窗口的步长
        self.input_ids = []   # 存储输入批次
        self.target_ids = []  # 存储目标批次
        # 把整个文本编码成token ID序列
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        # 核心：用滑动窗口切分文本，生成训练对
        for i in range(0, len(token_ids)-max_length, stride): # range(起始位置, 遍历终止位置, 步长)
            input_chunk = token_ids[i: i+max_length]       # 输入序列：从i开始，取'窗口大小'个词元                  [0,4]
            target_chunk = token_ids[i+1: i+max_length+1]  # 目标序列：从i+1开始，取'窗口大小'个词元（和输入错位1位）   [1,5]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk)) # 转成张量，存入列表

    def __len__(self):
        return len(self.input_ids)  # 返回数据集的总行数

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]  # 根据索引返回数据集的指定行

In [264]:
# 生成训练数据加载器的工具函数
def create_dataloader_v1(
    txt,             # 原始文本字符串
    batch_size=4,    # 句子数量（批次数量）
    max_length=256,  # 单句长度（窗口长度）
    stride=128,      # 滑动窗口的步长（举例：序列1是[0-255]，序列2是[128-383]，中间128个token重叠）
    shuffle=True,    # 是否打乱数据顺序
    drop_last=True,  # 是否丢弃最后一个不完整的批次
    num_workers=0    # 加载数据的线程数（Windows下默认0避免报错）
):
    tokenizer = tiktoken.get_encoding("gpt2")  # 初始化GPT-2用的tiktoken分词器
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)  # 用前面定义的GPTDatasetV1构建数据集
    # 用PyTorch的DataLoader封装数据集，生成训练用的数据加载器
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,   # 每次取多少个样本
        shuffle=shuffle,         # 每个epoch前是否打乱数据顺序
        drop_last=drop_last,     # 如果最后一个批次样本数不足batch_size，是否丢弃
        num_workers=num_workers  # 数据加载的后台线程数
    )
    return dataloader

- 让我们使用批次大小为1、上下文大小为4的设置，测试数据加载器在大语言模型（LLM）中的表现。

In [265]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [304]:
dataloader = create_dataloader_v1(raw_text, batch_size=2, max_length=4, stride=1, shuffle=False)
print('样本总数:',len(dataloader.dataset))  # 样本总数（跟步幅走，受滑动多少次决定）  数据来源:len(token_ids)-max_length: 5145-4=5141（当stride=1时）
print('批次总数:',len(dataloader))  # 批次总数 = 样本总数/batch_size（受drop_last影响）
data_iter = iter(dataloader)   # 把 DataLoader 包装成一个迭代器，方便用 next() 逐个取批次
first_batch = next(data_iter)  # 取出迭代器的下一个批次，这里就是第一个批次
print('第一批次:',first_batch) # 输出的 first_batch 是一个元组 (输入张量, 目标张量)

样本总数: 5141
批次总数: 2570
第一批次: [tensor([[  40,  367, 2885, 1464],
        [ 367, 2885, 1464, 1807]]), tensor([[ 367, 2885, 1464, 1807],
        [2885, 1464, 1807, 3619]])]


In [302]:
second_batch = next(data_iter) # 取下一个批次，验证滑动窗口
print('第二批次:',second_batch)

第二批次: [tensor([[2885, 1464, 1807, 3619],
        [1464, 1807, 3619,  402]]), tensor([[1464, 1807, 3619,  402],
        [1807, 3619,  402,  271]])]


- 下面是一个示例，步幅等于上下文长度（此处为4）：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/14.webp" width="500px">

- 我们还可以批量输出。
- 因为过多的重叠可能导致过拟合,这里增加了步幅。

In [317]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)  # 可自己调试

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("输入:\n", inputs)
print("目标:\n", targets)

print(len(dataloader.dataset)) # 样本总数
print(len(dataloader))         # 批次总数

输入:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
目标:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])
1286
160


## 2.7 创建词元嵌入(Creating token embeddings)

- 数据库快被准备好用于大语言模型（LLM）。
- 最后，让我们使用嵌入层将标记转换为连续的向量表示。
- 通常，这些嵌入层是大语言模型的一部分，并在模型训练过程中进行更新（训练）。

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/15.webp" width="400px">

- 假设我们有以下四个输入示例，经过分词后对应的输入 ID 为 2、3、5、1：

In [305]:
input_ids = torch.tensor([2, 3, 5, 1])

- 为了简化问题，假设我们只有一个包含 6 个词的小型词汇表，并且我们希望创建大小为 3 的嵌入。

In [306]:
vocab_size = 6  # 嵌入层需要支持的词汇表总数
output_dim = 3  # 嵌入向量的维度

torch.manual_seed(123)  # 用于设置随机数生成器的种子，确保结果可复现性
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)  # 结果是个6*3的矩阵, 每行表示一个词的嵌入向量

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


- 对于熟悉（one-hot encoding）的人来说，上述嵌入层方法本质上只是实现one-hot编码后接矩阵乘法的更高效方式，具体实现可以参考[./embedding_vs_matmul](../03_bonus_embedding-vs-matmul)中的补充代码。
- 嵌入层只是对one-hot encoding和矩阵乘法方法的高效实现，因此可以将其视为一个神经网络层，并通过反向传播进行优化。

- 将其应用到一个词元 ID 上, 以获取嵌入向量

In [307]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


- 请注意，上述内容是 `embedding_layer` 权重矩阵中的第四行。
- 要嵌入上述所有四个 `input_ids` 值，我们执行以下操作：

In [308]:
print(embedding_layer(input_ids))  # 代入[2, 3, 5, 1]

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


- 嵌入层本质上是一个查找操作：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/16.webp?123" width="500px">

- **您可能对比较嵌入层与常规线性层的附加内容感兴趣：[../03_bonus_embedding-vs-matmul](../03_bonus_embedding-vs-matmul)**

## 2.8 位置信息编码

- 嵌入层将 ID 转换为相同的向量表示

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/17.webp" width="400px">

- 位置信息与标记向量结合，形成大语言模型的最终输入：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/18.webp" width="500px">

- 字节对编码器的词汇表大小为 50,257：
- 假设我们想将输入标记编码为一个 256 维的向量表示：

In [309]:
vocab_size = 50257  # GPT-2 BPE分词器的词汇表大小
output_dim = 256    # 词向量维度：每个token会被转成256维的向量表示
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
# 本质上，这是一个可学习的权重矩阵，形状是 [50257, 256]，每个 token ID（0~50256）对应矩阵里的一行，这一行就是该 token 的词向量
# 核心作用：把离散的 token ID，转换成模型能处理的连续向量，解决 “词的语义表示” 问题
print(token_embedding_layer.weight.shape)

torch.Size([50257, 256])


- 如果我们从数据加载器中采样数据，我们将每个批次中的词元嵌入为一个 256 维的向量。
- 如果我们有一个批次大小为 8，每个批次包含 4 个词元，这将得到一个 8 x 4 x 256 的张量：

In [318]:
max_length = 4  # 每个批次4个词元
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=max_length,stride=max_length, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("词元ID:\n", inputs)
print("形状:\n", inputs.shape)

词元ID:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
形状:
 torch.Size([8, 4])


In [319]:
token_embeddings = token_embedding_layer(inputs)  # 把每个 token ID，通过嵌入层的权重矩阵查表，转换成 256 维的词向量
print(token_embeddings.shape)  # 形状变化：从 [8, 4] → [8, 4, 256]，也就是每个 token ID 都被转换成了 256 维的语义向量

torch.Size([8, 4, 256])


- GPT-2 使用绝对位置嵌入，因此我们只需要创建另一个位置嵌入层：
- 嵌入层本质上是一个查找表,大小为(context_length, output_dim)  (4,256)

In [323]:
context_length = max_length  # 每句的词数（max_length=4）
# 位置嵌入层：这是一个 [4, 256] 的可学习矩阵，每个位置（第0、1、2、3位）对应一行位置向量
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)  # output_dim：和词向量维度保持一致（都是 256），这样后面才能直接相加
pos_embeddings = pos_embedding_layer(torch.arange(max_length))  # torch.arange(max_length)：生成批次的位置索引 [0, 1, 2, 3]，表示每个 token 在批次中的位置
# 核心作用：给每句的每个词分配一个唯一的可学习向量，解决「同一个 token 在不同位置无法区分」的问题
print(pos_embeddings.shape)

torch.Size([4, 256])


- 为了创建大语言模型（LLM）中使用的输入嵌入，我们只需将标记嵌入和位置嵌入相加：

In [324]:
input_embeddings = token_embeddings + pos_embeddings # 广播机制: [8, 4, 256] + [4, 256]
# 结果含义：每个 token 的向量，现在同时包含了词的语义信息（来自词嵌入）和位置信息（来自位置嵌入），模型就能区分「句子开头的词」和「句子结尾的词」了
print(input_embeddings.shape)

torch.Size([8, 4, 256])


- 在输入处理流程的初始阶段，输入文本被分割为独立的标记。
- 在此分割后，这些标记根据预定义的单词表转换为标记 ID：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/19.webp" width="400px">

# 总结与收获

请参见 [./dataloader.ipynb](./dataloader.ipynb) 代码笔记本，这是我们在本章中实现的数据加载器的简洁版，并将在后续章节中用于训练 GPT 模型。

请参见 [./exercise-solutions.ipynb](./exercise-solutions.ipynb) 获取习题解答。